In [1]:
# === SESSION BOOTSTRAP ===
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Mounted at /content/drive
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install pandas numpy scipy --quiet

In [3]:
# Bootstrap 95% CIs on the three headline claims, each with the CORRECT resampling unit.
import numpy as np, pandas as pd, glob, os
from scipy.stats import spearmanr
B=10000; rng=np.random.default_rng(20260716)
def ci(vals,lo=2.5,hi=97.5):
    v=np.asarray([x for x in vals if x==x]); return float(np.percentile(v,lo)),float(np.percentile(v,hi))
print("bootstrap B =",B)

bootstrap B = 10000


In [4]:
# ---------- (1) fragility-coverage Spearman: resample FAMILIES (family is the unit) ----------
# Per-family (fragility_raw, shift marginal coverage) means, taken directly from Tables 3, 6, and the UAV-CAS fragility.
FAM={
 "UAVIDS-2025":[("Sybil",2.43,0.912),("Flooding",1.17,0.718),("Wormhole",3.04,0.635),("Blackhole",5.11,0.408)],
 "UAV-NIDD":[("UDP Flooding",0.00,0.802),("BruteForce",0.16,0.656),("De-auth",2.31,0.285),
             ("MITM",0.46,0.151),("DDoS",5.51,0.065),("Jamming",4.71,0.001)],
 "UAV-CAS":[("Blackhole",0.611,0.557),("DDoS",0.001,0.999),("DoS",0.000,0.998),("Replay",0.210,0.879),("Wormhole",0.634,0.546)],
}
rows1=[]
for ds,fam in FAM.items():
    fr=np.array([f[1] for f in fam]); cv=np.array([f[2] for f in fam]); n=len(fr)
    point=spearmanr(fr,cv).correlation; boot=[]
    for _ in range(B):
        idx=rng.integers(0,n,n)
        if len(np.unique(fr[idx]))<2 or len(np.unique(cv[idx]))<2: continue
        boot.append(spearmanr(fr[idx],cv[idx]).correlation)
    lo,hi=ci(boot)
    rows1.append({"dataset":ds,"n_families":n,"spearman":round(point,3),"CI_low":round(lo,3),"CI_high":round(hi,3)})
    print(f"{ds:12s} Spearman {point:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  (resampling {n} families)")
pd.DataFrame(rows1).to_csv("reports/20_ci_fragility_spearman.csv",index=False)
print("\nNote: intervals are wide because only 4-6 families exist per dataset; this is the honest precision.")

UAVIDS-2025  Spearman -0.800  95% CI [-1.000, +1.000]  (resampling 4 families)
UAV-NIDD     Spearman -0.886  95% CI [-1.000, -0.200]  (resampling 6 families)
UAV-CAS      Spearman -0.900  95% CI [-1.000, -0.111]  (resampling 5 families)

Note: intervals are wide because only 4-6 families exist per dataset; this is the honest precision.


In [5]:
# ---------- (2) key coverage collapses: resample SEEDS ----------
def find(*pats):
    for p in pats:
        g=glob.glob("reports/"+p)
        if g: return g[0]
    return None
uni=find("10_unified_panel_raw.csv","*unified_panel_raw.csv","10_*panel*raw.csv")
cas=find("14_uavcas_panel_raw.csv","*uavcas_panel_raw.csv","14_*panel*raw.csv")
print("panel files:",uni,cas)
def seed_ci(csv,dsf,fam):
    d=pd.read_csv(csv)
    if dsf is not None and "dataset" in d.columns: d=d[d["dataset"]==dsf]
    v=d[d["held_out"].astype(str)==fam]["shift_cov_marg"].values
    if len(v)==0: return None
    boot=[np.mean(rng.choice(v,len(v),replace=True)) for _ in range(B)]; lo,hi=ci(boot)
    return float(np.mean(v)),lo,hi,len(v)
rows2=[]
targets=[(uni,"UAV-NIDD","Jamming"),(uni,"UAV-NIDD","DDoS"),(uni,"UAVIDS-2025","Blackhole Attack"),
         (cas,None,"Blackhole"),(cas,None,"Wormhole")]
for csv,ds,fam in targets:
    if csv is None: print(f"[skip] {ds} {fam}: panel csv missing"); continue
    r=seed_ci(csv,ds,fam)
    if r is None: print(f"[skip] {ds} {fam}: family not found in {csv}"); continue
    m,lo,hi,ns=r; rows2.append({"dataset":ds or "UAV-CAS","family":fam,"coverage":round(m,3),"CI_low":round(lo,3),"CI_high":round(hi,3),"n_seeds":ns})
    print(f"{(ds or 'UAV-CAS'):12s} {fam:18s} coverage {m:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (n={ns} seeds)")
pd.DataFrame(rows2).to_csv("reports/20_ci_coverage_collapse.csv",index=False)

panel files: reports/10_unified_panel_raw.csv reports/14_uavcas_panel_raw.csv
UAV-NIDD     Jamming            coverage 0.001  95% CI [0.000, 0.002]  (n=10 seeds)
UAV-NIDD     DDoS               coverage 0.065  95% CI [0.029, 0.114]  (n=10 seeds)
UAVIDS-2025  Blackhole Attack   coverage 0.408  95% CI [0.408, 0.409]  (n=10 seeds)
UAV-CAS      Blackhole          coverage 0.557  95% CI [0.552, 0.563]  (n=10 seeds)
UAV-CAS      Wormhole           coverage 0.546  95% CI [0.540, 0.552]  (n=10 seeds)


In [6]:
# ---------- (3) TSFS gains over controls: paired bootstrap over SEEDS ----------
# locate the TSFS four-way per-seed raw file by pattern (repo is cloned in Colab)
tsfs=find("*tsfs*raw*.csv","*fourway*raw*.csv","11_*raw*.csv","*baselines*raw*.csv","*tsfs*.csv")
print("TSFS raw candidate:",tsfs)
rows3=[]
if tsfs is None:
    print("[skip] TSFS four-way per-seed raw not found; list reports/ to find it:", glob.glob("reports/11_*")+glob.glob("reports/*tsfs*"))
else:
    d=pd.read_csv(tsfs); print("columns:",list(d.columns)); cm={c.lower():c for c in d.columns}
    def C(*names):
        for nm in names:
            if nm in cm: return cm[nm]
    ct=C("tsfs","fragility_drop","tsfs_cov"); crd=C("random","random_drop","rand"); cli=C("low_imp","lowimp","low_importance","lowest_imp","lowimp_drop"); ch=C("held_out","family"); cs=C("seed")
    if None in (ct,crd,cli,ch,cs):
        print("[skip] expected columns not all present. Have:",list(d.columns))
    else:
        fams=sorted(d[ch].astype(str).unique())+["__MEAN__"]
        for fam in fams:
            sub=d if fam=="__MEAN__" else d[d[ch].astype(str)==fam]
            if fam=="__MEAN__":
                g=sub.groupby(cs).agg(t=(ct,"mean"),r=(crd,"mean"),l=(cli,"mean")); t,r_,l=g["t"].values,g["r"].values,g["l"].values
            else: t,r_,l=sub[ct].values,sub[crd].values,sub[cli].values
            if len(t)<2: continue
            dr=t-r_; dl=t-l; ns=len(t)
            br=[np.mean(rng.choice(dr,ns,replace=True)) for _ in range(B)]; bl=[np.mean(rng.choice(dl,ns,replace=True)) for _ in range(B)]
            rlo,rhi=ci(br); llo,lhi=ci(bl)
            rows3.append({"family":("Mean (10 families)" if fam=="__MEAN__" else fam),
                "gain_vs_random":round(float(dr.mean()),3),"rand_CI":f"[{rlo:+.3f}, {rhi:+.3f}]",
                "gain_vs_lowimp":round(float(dl.mean()),3),"lowimp_CI":f"[{llo:+.3f}, {lhi:+.3f}]","n_seeds":ns})
        T=pd.DataFrame(rows3); print(T.to_string(index=False)); T.to_csv("reports/20_ci_tsfs_gains.csv",index=False)

TSFS raw candidate: reports/11_tsfs_vs_baselines_raw.csv
columns: ['dataset', 'held_out', 'seed', 'nfeat', 'k_dropped', 'rho_int_oracle', 'topk_jaccard', 'cov_baseline', 'ba_baseline', 'cov_tsfs', 'ba_tsfs', 'cov_random', 'ba_random', 'cov_low_imp', 'ba_low_imp']
[skip] expected columns not all present. Have: ['dataset', 'held_out', 'seed', 'nfeat', 'k_dropped', 'rho_int_oracle', 'topk_jaccard', 'cov_baseline', 'ba_baseline', 'cov_tsfs', 'ba_tsfs', 'cov_random', 'ba_random', 'cov_low_imp', 'ba_low_imp']


In [7]:
print("\n================ BOOTSTRAP 95% CI SUMMARY ================")
for f,label in [("reports/20_ci_fragility_spearman.csv","(1) fragility-coverage Spearman  [resampled over FAMILIES]"),
                ("reports/20_ci_coverage_collapse.csv","(2) key coverage collapses  [resampled over seeds]"),
                ("reports/20_ci_tsfs_gains.csv","(3) TSFS gains over controls  [paired, over seeds]")]:
    if os.path.exists(f): print("\n"+label); print(pd.read_csv(f).to_string(index=False))


================ BOOTSTRAP 95% CI SUMMARY ================

(1) fragility-coverage Spearman  [resampled over FAMILIES]
    dataset  n_families  spearman  CI_low  CI_high
UAVIDS-2025           4    -0.800    -1.0    1.000
   UAV-NIDD           6    -0.886    -1.0   -0.200
    UAV-CAS           5    -0.900    -1.0   -0.111

(2) key coverage collapses  [resampled over seeds]
    dataset           family  coverage  CI_low  CI_high  n_seeds
   UAV-NIDD          Jamming     0.001   0.000    0.002       10
   UAV-NIDD             DDoS     0.065   0.029    0.114       10
UAVIDS-2025 Blackhole Attack     0.408   0.408    0.409       10
    UAV-CAS        Blackhole     0.557   0.552    0.563       10
    UAV-CAS         Wormhole     0.546   0.540    0.552       10


In [ ]:
!git add reports/ notebooks/
!git commit -m "20 bootstrap 95% CIs: fragility-coverage Spearman (family-resampled), coverage collapses and TSFS gains (seed-resampled)"
!git push origin main